Make NumPy available:

In [57]:
import numpy as np

## Exercise 07.1 (indexing and timing)

Create two very long NumPy 1D-arrays (vectors) `x` and `y` and sum the arrays using:

1. The NumPy addition syntax, `z = x + y`; and
2. A `for` loop that computes the sum entry-by-entry

Compare the time required for the two approaches for arrays of different lengths (use a very long array for 
the timing). The values of the array entries are not important for this test. Use `%time` to report the time.

*Hint:* To loop over an array using indices, try a construction like:

In [58]:
x = np.ones(10)
y = np.ones(len(x))
for i in range(len(x)):
    print(x[i]*y[i])

1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0


#### (1) Add two arrays using built-in addition operator:

In [59]:
import random

# Initialise the two long vectors
x = np.random.rand(1000000)
y = np.random.rand(1000000)

# Print time taken to finish the calculations
%time z = x + y


CPU times: user 11.6 ms, sys: 41.1 ms, total: 52.7 ms
Wall time: 120 ms


#### (2) Add two arrays using own implementation:

In [60]:
# Define a list for looping through the arrays
def add_loop(x,y):
    z = []
    for i in range(len(x)):
       z.append(x[i] + y[i])
    return z

# Find time taken to execute the function
%time z = add_loop(x, y)

CPU times: user 186 ms, sys: 27.6 ms, total: 213 ms
Wall time: 244 ms


### Optional extension: just-in-time (JIT) compilation

You will see a large difference in the time required between your NumPy and 'plain' Python implementations. This is due to Python being an *interpreted* language as opposed to a *compiled* language. A way to speed up plain Python implementions is to convert the interpreted Python code into compiled code. A tool for doing this is [Numba](https://numba.pydata.org/).

Below is an example using Numba and JIT to accelerate a computation:

In [61]:
%pip -q install numba 
import numba
import math

def compute_sine_native(x):
    z = np.zeros(len(x))
    for i in range(len(z)):
        z[i] = math.sin(x[i])
    return z

@numba.jit
def compute_sine_jit(x):
    z = np.zeros(len(x))
    for i in range(len(z)):
        z[i] = math.sin(x[i])
    return z
    
x = np.ones(10000000)
%time z = compute_sine_native(x)
compute_sine_jit(x)
%time z = compute_sine_jit(x)


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
CPU times: user 1.59 s, sys: 149 ms, total: 1.74 s
Wall time: 2.3 s
CPU times: user 55.8 ms, sys: 4.05 ms, total: 59.8 ms
Wall time: 68.7 ms


**Task:** Test if Numba can be used to accelerate your implementation that uses indexing to sum two arrays, and by how much.

In [62]:
import numba
import numpy as np

x = np.random.rand(1000000)
y = np.random.rand(1000000)

# Use Numba for computation
@numba.jit
def add_jit(x, y):
    z = []
    for i in range(len(x)):
       z.append(x[i] + y[i])
    return z

%time z = add_jit(x, y)

CPU times: user 99.9 ms, sys: 17.9 ms, total: 118 ms
Wall time: 141 ms


## Exercise 07.2 (member functions and slicing)

Anonymised scores (out of 60) for an examination are stored in a NumPy array. Write:

1. A function that takes a NumPy array of the raw scores and returns the scores as percentages, sorted from 
   lowest to highest (try using `scores.sort()`, where `scores` is a NumPy array holding the scores).
1. A function that returns the maximum, minimum and mean of the raw scores as a dictionary with the 
   keys '`min`', '`max`' and '`mean`'. Use the NumPy array functions `min()`, `max()` and `mean()` to do the 
   computation, e.g. `max = scores.max()`.  
   
   Design your function for the min, max and mean to optionally exclude the highest and lowest scores from the 
   computation of the min, max and mean. 
   
   *Hint:* sort the array of scores and use array slicing to exclude
   the first and the last entries.

Use the scores 
```python
scores = np.array([58.0, 35.0, 24.0, 42, 7.8])
```
to test your functions.

In [63]:
def to_percentage_and_sort(scores):
    # Find the decimal division results and time by 100 to find percentage
    scores = scores / 60 * 100
    scores.sort()
    return scores

def statistics(scores, exclude=False):
    scores.sort()

    # Check if the calculation requires maximum and minimum scores to be excluded
    if (exclude is True and len(scores) > 2):
        # Use slicing to exclude the first and last element
        scores = scores[1:-1]
        print(scores)
        
    return {"min": scores.min(), "max": scores.max(), "mean": scores.mean()}
    

In [64]:
## tests ##

scores = np.array([35.0, 58.0, 24.0, 42, 7.8])
assert np.isclose(to_percentage_and_sort(scores), [ 13.0, 40.0, 58.33333333,  70.0, 96.66666667]).all()

s0 = statistics(scores)
assert np.isclose(s0["min"], 7.8)
assert np.isclose(s0["mean"], 33.36)
assert np.isclose(s0["max"], 58.0)

s1 = statistics(scores, True)
assert np.isclose(s1["min"], 24.0)
assert np.isclose(s1["mean"], 33.666666666666666667)
assert np.isclose(s1["max"], 42.0)

[24. 35. 42.]


## Exercise 07.3 (slicing)

For the two-dimensional array

In [65]:
A = np.array([[4.0, 7.0, -2.43, 67.1],
             [-4.0, 64.0, 54.7, -3.33],
             [2.43, 23.2, 3.64, 4.11],
             [1.2, 2.5, -113.2, 323.22]])
print(A)

[[   4.      7.     -2.43   67.1 ]
 [  -4.     64.     54.7    -3.33]
 [   2.43   23.2     3.64    4.11]
 [   1.2     2.5  -113.2   323.22]]


use array slicing for the below operations, printing the results to the screen to check. Try to use array slicing such that your code would still work if the dimensions of `A` were enlarged.

**Check your results carefully against hand computations.**

#### 1. Extract the third column as a 1D array

In [73]:
print(A[:,2])

[  -2.43   54.7     3.64 -113.2 ]


#### 2. Extract the first two rows as a 2D sub-array

In [67]:
print(A[0:2])

[[ 4.    7.   -2.43 67.1 ]
 [-4.   64.   54.7  -3.33]]


#### 3.  Extract the bottom-right $2 \times 2$ block as a 2D sub-array

In [68]:
print(A[2:,2:])

[[   3.64    4.11]
 [-113.2   323.22]]


#### 4. Sum the last column

In [74]:
print(np.sum(A[:, -1]))

391.1


#### Compute transpose

Compute the transpose of `A` (search online to find the function/syntax to do this).

In [70]:
print(np.transpose(A))

[[   4.     -4.      2.43    1.2 ]
 [   7.     64.     23.2     2.5 ]
 [  -2.43   54.7     3.64 -113.2 ]
 [  67.1    -3.33    4.11  323.22]]


## Exercise 07.4 (optional extension)

In a previous exercise (04.5) you implemented the bisection algorithm to find approximate roots of a mathematical function. Use the SciPy bisection function `optimize.bisect` (https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.bisect.html) to find roots of the mathematical function that was used in the previous exercise. Compare the results computed by SciPy and your program from the earlier exercise, and compare the computational time (using `%time`).

In [71]:
from scipy import optimize

def my_f(x):
    """Evaluate polynomial function"""
    return x**5 / 10 + x**3 - 10 * x**2 + 4 * x + 7

# Finds root using iteration
def compute_root(f, x0, x1, tol, max_it):
    """Compute roots of a function using bisection"""
    x_mid = (x0 + x1) / 2
    f_cur = f(x_mid)
    it = 1
    
    while not abs(f_cur) < tol and it < max_it:
        it += 1
        # Calculate new mid using current upper and lower limit
        x_mid = (x0 + x1) / 2
        f_cur = f(x_mid)

        # If the current mid is below x-axis, decrease upper limit
        if f_cur < 0:
            x1 = x_mid
        # If the current mid is above x-axis, increase lower limit
        elif f_cur > 0:
            x0 = x_mid
        
    f_mid = f_cur
     
    return x_mid, f_mid, it



In [72]:
%time print(compute_root(my_f, 0, 2, 1.0e-6, 1000))
%time print(optimize.bisect(my_f, 0, 2))

(1.1568354368209839, 3.9180508970559913e-07, 25)
CPU times: user 84 μs, sys: 14 μs, total: 98 μs
Wall time: 91.1 μs
1.1568354643623024
CPU times: user 136 μs, sys: 508 μs, total: 644 μs
Wall time: 1.87 ms
